In [ ]:
# ============================================================
# 🥷 AIBO v7.4.0 ワンプッシュ起動セル
# Phase 1: HF cache · Phase 2: 依存 · Phase 3: 起動 · Phase 4: 完成 · Phase 5: API 公開
# ============================================================
import os, sys, importlib, subprocess, shutil, time

# Phase 0: import warm-up は無効化 (VRAM OOM の原因となるため)
# torch/nunchaku を background thread で import すると CUDA コンテキスト初期化が
# 競合し、main thread のメモリプール確保が失敗するケースを確認。
# 4 秒の最適化を捨てて VRAM 安定性を優先。

# ─── Phase 1: Drive mount + HF cache ─────────────────────────
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive", force_remount=False)
# 二段キャッシュ: Drive (永続) → /content NVMe (高速 1-2 GB/s) → VRAM
# Drive 帯域 (40-100 MB/s) を経由せず NVMe から VRAM 直送することで I/O ボトルネックを回避
GDRIVE_HF_CACHE = "/content/drive/MyDrive/aibo_hf_cache"
CONTENT_HF_CACHE = "/content/aibo_hf_cache"
os.makedirs(GDRIVE_HF_CACHE, exist_ok=True)
os.makedirs(CONTENT_HF_CACHE, exist_ok=True)

CACHE_SYNC_TIMEOUT = 180  # seconds

def _du_bytes(path):
    try:
        return int(subprocess.check_output(["du", "-sb", path], stderr=subprocess.DEVNULL).split()[0])
    except Exception:
        return 0

t_sync = time.time()
drive_size = _du_bytes(GDRIVE_HF_CACHE)
content_size = _du_bytes(CONTENT_HF_CACHE)
print(f"📦 Drive cache: {drive_size/1e9:.1f}GB · /content cache: {content_size/1e9:.1f}GB")

# /content cache が Drive cache の 95% に満たない場合のみ同期
if content_size < drive_size * 0.95 and drive_size > 0:
    print(f"⏳ Drive → /content 二段キャッシュ同期中 ({drive_size/1e9:.1f}GB, timeout={CACHE_SYNC_TIMEOUT}s)...")
    _sync_ok = False
    # Method 1: tar pipe (rsync 廃止 · Drive FUSE ハング回避)
    try:
        _cmd = f'tar cf - -C "{GDRIVE_HF_CACHE}" . | tar xf - -C "{CONTENT_HF_CACHE}"'
        _r = subprocess.run(["bash", "-c", _cmd], capture_output=True, text=True, timeout=CACHE_SYNC_TIMEOUT)
        if _r.returncode == 0:
            _sync_ok = True
            print(f"✅ 二段キャッシュ同期完了 (tar): {time.time()-t_sync:.1f}s")
        else:
            print(f"⚠️ tar pipe 失敗: {(_r.stderr or '')[-200:]}")
    except subprocess.TimeoutExpired:
        print(f"⚠️ tar pipe timeout ({CACHE_SYNC_TIMEOUT}s)")
    except Exception as _e:
        print(f"⚠️ tar pipe 例外: {_e}")
    # Method 2: symlink fallback (Drive 直結 · NVMe 高速化なし)
    if not _sync_ok:
        print("🔗 symlink fallback: Drive 直結に切替 (NVMe 高速化なし)")
        shutil.rmtree(CONTENT_HF_CACHE, ignore_errors=True)
        os.symlink(GDRIVE_HF_CACHE, CONTENT_HF_CACHE)
        print(f"✅ symlink: {CONTENT_HF_CACHE} → {GDRIVE_HF_CACHE}")
else:
    print(f"✅ /content cache 既存利用 (sync スキップ)")

# /root/.cache/huggingface → /content/aibo_hf_cache symlink
# (env 変数を無視する library 対策、全 HF DL を NVMe に固定)
ROOT_HF = "/root/.cache/huggingface"
os.makedirs("/root/.cache", exist_ok=True)
if os.path.islink(ROOT_HF) or os.path.exists(ROOT_HF):
    subprocess.run(["rm", "-rf", ROOT_HF], check=False)
os.symlink(CONTENT_HF_CACHE, ROOT_HF)
print(f"✅ HF cache symlink: {ROOT_HF} → {CONTENT_HF_CACHE}")

# env 変数も /content (NVMe) を指す
for k in ["HF_HOME", "TRANSFORMERS_CACHE", "HUGGINGFACE_HUB_CACHE", "HF_HUB_CACHE"]:
    os.environ[k] = CONTENT_HF_CACHE
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
print(f"✅ HF cache → {CONTENT_HF_CACHE} (NVMe)")

# ─── Phase 2: torchsde 事前 install ─────────────────────────
for dep in ['torchsde']:
    try:
        __import__(dep)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", dep], check=True)

# ─── Phase 3: AIBO 起動 ─────────────────────────────────────
print("\n" + "=" * 60)
print("🚀 AIBO v7.4.0 起動シーケンス")
print("=" * 60)

AIBO_ROOT = "/content/drive/MyDrive/aibo_v7"

# BOM 除去
for fname in ["01_config.py", "02_colab_setup.py", "03_identity_engine.py",
              "04_pipeline_manager.py", "05_orchestrator.py", "06_ui.py",
              "07_main.py", "08_face_refiner.py"]:
    fpath = os.path.join(AIBO_ROOT, fname)
    with open(fpath, "rb") as f:
        c = f.read()
    if c.startswith(b'\xef\xbb\xbf'):
        with open(fpath, "wb") as f:
            f.write(c[3:])

if AIBO_ROOT not in sys.path:
    sys.path.insert(0, AIBO_ROOT)

# モジュール import (キャッシュクリア)
for m in list(sys.modules):
    if m.startswith(("01_","02_","03_","04_","05_","06_","07_","08_")):
        del sys.modules[m]

t_imp = time.perf_counter()
mod01 = importlib.import_module("01_config")
mod02 = importlib.import_module("02_colab_setup")  # numpy 自動チェック + 必要時のみ自動再起動
mod03 = importlib.import_module("03_identity_engine")
mod04 = importlib.import_module("04_pipeline_manager")
mod05 = importlib.import_module("05_orchestrator")
mod06 = importlib.import_module("06_ui")
mod07 = importlib.import_module("07_main")
mod08 = importlib.import_module("08_face_refiner")
print(f"⏱️ import: {time.perf_counter() - t_imp:.2f}s")

SystemConfig = mod01.SystemConfig
GenerationConfig = mod01.GenerationConfig
IdentityConfig = mod01.IdentityConfig
StudioMode = mod01.StudioMode
AiboMain = mod07.AiboMain

# AiboMain 起動
t_boot = time.perf_counter()
aibo = AiboMain()
if hasattr(aibo, "run"):
    aibo.run(enable_gradio=False)  # A方式運用 · Phase G スキップ
print(f"⏱️ AiboMain 起動: {time.perf_counter() - t_boot:.2f}s")

orchestrator = aibo.orchestrator
pm = orchestrator.pm
ie = orchestrator.ie
ipa = ie.ip_adapter
gen_cfg = GenerationConfig()
id_cfg = IdentityConfig()

# ─── Phase 4: Phase 1 完成状態セットアップ ──────────────────
print("\n" + "=" * 60)
print("🔧 Phase 1 完成状態セットアップ")
print("=" * 60)
tf = pm._shared_transformer
if pm.pipe_cnet is None:
    pm.ensure_controlnet()
pm._cn_forward_wrapped = False
pm._wrap_transformer_forward_for_cn()
if pm.pipe_cnet.image_encoder is None:
    pm.pipe_cnet.image_encoder = ipa._image_encoder
    pm.pipe_cnet.feature_extractor = ipa._feature_extractor
ipa.set_scale(pm.pipe_base, id_cfg.ip_adapter_weight)
ipa.set_scale(pm.pipe_cnet, id_cfg.ip_adapter_weight)
print(f"✅ forward={tf.forward.__qualname__}")
print(f"✅ IP-Adapter scale={id_cfg.ip_adapter_weight}")

print("\n" + "=" * 60)
print("🎉 v7.4.0 Phase 1 完成 · 即生成可能!")
print("=" * 60)
import sys as _sys
_sys.stdout.flush()

# ─── Phase 5: FastAPI + ngrok 公開 ───────────────────────────
import traceback as _tb
_sys.stdout.flush()
print("\n" + "=" * 60, flush=True)
print("🚀 FastAPI + ngrok 公開", flush=True)
print("=" * 60, flush=True)

try:
    import threading, requests
    for pkg in ["fastapi", "pyngrok"]:
        try:
            __import__(pkg)
        except ImportError:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

    from pyngrok import ngrok, conf
    from google.colab import userdata

    srv = importlib.import_module("09_fastapi_server")

    # 既起動チェック
    already_running = False
    try:
        r = requests.get("http://localhost:8000/api/system/status", timeout=2)
        if r.status_code == 200 and r.json().get("orchestrator_attached"):
            already_running = True
            print("ℹ️ FastAPI 既起動、スキップ", flush=True)
    except: pass

    if not already_running:
        srv.attach_orchestrator(orchestrator, pm)
        threading.Thread(
            target=lambda: srv.run_server(host="0.0.0.0", port=8000, log_level="warning"),
            daemon=True, name="aibo-fastapi"
        ).start()
        time.sleep(3)

    r = requests.get("http://localhost:8000/api/system/status", timeout=5)
    b = r.json()
    print(f"✅ FastAPI · GPU={b['gpu_name']} · VRAM={b['vram_used_gb']:.1f}/{b['vram_total_gb']:.1f} GB", flush=True)

    # ngrok
    conf.get_default().auth_token = userdata.get('NGROK_AUTH_TOKEN')
    try:
        for t in ngrok.get_tunnels():
            ngrok.disconnect(t.public_url)
        ngrok.kill()
        time.sleep(1)
    except: pass

    public_url = ngrok.connect(8000, "http").public_url
    time.sleep(3)

    # ngrok 疎通確認 (リトライ付き)
    for i in range(5):
        try:
            r = requests.get(f"{public_url}/api/system/status",
                             headers={"ngrok-skip-browser-warning": "true"}, timeout=10)
            if r.status_code == 200:
                break
        except: pass
        time.sleep(2)

    env_content = f"NEXT_PUBLIC_API_URL={public_url}\nNEXT_PUBLIC_NGROK_SKIP_WARNING=true"
    with open(f"{AIBO_ROOT}/.env.local.latest.txt", "w") as f:
        f.write(env_content)

    print(f"\n🔌 API URL: {public_url}", flush=True)
    print("\n" + "=" * 60, flush=True)
    print("📋 PC の .env.local にコピー", flush=True)
    print("=" * 60, flush=True)
    print(env_content, flush=True)
    print("\n" + "=" * 60, flush=True)
    print("🌐 ブラウザで開く (PC で npm run dev が走ってる前提)", flush=True)
    print("=" * 60, flush=True)
    print("http://localhost:3000", flush=True)
    print("\n🎉 ワンプッシュ起動完了 🥷", flush=True)
except Exception as _e:
    print(f"\n❌ Phase 5 で例外発生: {type(_e).__name__}: {_e}", flush=True)
    _tb.print_exc()
    raise

def sync_back_to_drive():
    """/content cache の内容を Drive に書き戻し (新規 DL の永続化用)"""
    if os.path.islink(CONTENT_HF_CACHE):
        print("ℹ️ symlink モード · Drive 直結のため sync 不要")
        return
    t = time.time()
    try:
        _cmd = f'tar cf - -C "{CONTENT_HF_CACHE}" . | tar xf - -C "{GDRIVE_HF_CACHE}"'
        subprocess.run(["bash", "-c", _cmd], capture_output=True, text=True, timeout=300)
        print(f"✅ Drive 同期完了 (tar): {time.time()-t:.1f}s")
    except subprocess.TimeoutExpired:
        print(f"⚠️ Drive 同期 timeout (300s) · 手動で再試行してください")
    except Exception as _e:
        print(f"⚠️ Drive 同期失敗: {_e}")

In [ ]:
# ============================================================
# 🔍 環境確認 (Cell 0 実行後 · 任意)
# ============================================================
import json
import urllib.request
from pathlib import Path

AIBO_ROOT = "/content/drive/MyDrive/aibo_v7"
print("=" * 60)
print("🔍 環境確認")
print("=" * 60)

# 1. FastAPI (local)
try:
    with urllib.request.urlopen("http://127.0.0.1:8000/api/system/status", timeout=10) as resp:
        st = json.loads(resp.read().decode())
    print("✅ localhost:8000")
    print(f"   orchestrator_attached: {st.get('orchestrator_attached')}")
    if st.get("gpu_available"):
        print(f"   GPU: {st.get('gpu_name')}")
        print(f"   VRAM: {st.get('vram_used_gb')}/{st.get('vram_total_gb')} GB ({st.get('vram_pct')}%)")
except Exception as exc:
    print(f"❌ localhost:8000 · {exc}")

# 2. .env.local.latest.txt + ngrok 疎通
env_file = Path(AIBO_ROOT) / ".env.local.latest.txt"
if env_file.is_file():
    text = env_file.read_text(encoding="utf-8").strip()
    print(f"\n✅ {env_file.name}:")
    print(text)
    ngrok_url = None
    for line in text.splitlines():
        if line.startswith("NEXT_PUBLIC_API_URL="):
            ngrok_url = line.split("=", 1)[1].strip()
            break
    if ngrok_url:
        try:
            req = urllib.request.Request(
                f"{ngrok_url.rstrip('/')}/api/system/status",
                headers={"ngrok-skip-browser-warning": "true"},
            )
            with urllib.request.urlopen(req, timeout=15) as resp:
                ext = json.loads(resp.read().decode())
            print(f"\n✅ ngrok 疎通 OK · orchestrator_attached={ext.get('orchestrator_attached')}")
        except Exception as exc:
            print(f"\n⚠️ ngrok 疎通: {exc}")
else:
    print(f"\n⚠️ {env_file} がありません (Cell 0 を先に実行)")
